# Vorhersage-Workflow mittels importierter Daten-Pipeline

Dieses Notebook nutzt die ausgelagerte `non_labled_data_pipeline`, um die Testdaten aufzubereiten. Der Fokus liegt hier auf dem Laden des Modells und der Erstellung der finalen Vorhersagen.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from catboost import CatBoostClassifier

# Importieren der Verarbeitungsfunktion aus Ihrer Pipeline-Datei
from pipeline.non_labled_data_pipeline import process_unlabeled_data

## Schritt 1: Laden des trainierten Modells und des Preprocessors

Wir laden die zuvor gespeicherten Artefakte: das trainierte CatBoost-Modell und den gefitteten `preprocessor`.

In [2]:
# --- Pfade definieren und neueste Dateien finden ---
weights_dir = Path('./weights')

# Finde die neueste Modelldatei (.cbm)
model_files = sorted([f for f in weights_dir.iterdir() if f.suffix == '.cbm'], reverse=True)
if not model_files:
    raise FileNotFoundError("Keine .cbm Modelldatei im Ordner ./weights gefunden.")
latest_model_path = model_files[0]

# Finde die neueste Preprocessor-Datei (.joblib)
preprocessor_files = sorted([f for f in weights_dir.iterdir() if f.suffix == '.joblib'], reverse=True)
if not preprocessor_files:
    raise FileNotFoundError("Keine .joblib Preprocessor-Datei im Ordner ./weights gefunden.")
latest_preprocessor_path = preprocessor_files[0]


# --- Modell und Preprocessor laden ---
print(f"Lade Modell von: {latest_model_path}")
loaded_model = CatBoostClassifier()
loaded_model.load_model(latest_model_path)

print(f"Lade Preprocessor von: {latest_preprocessor_path}")
loaded_preprocessor = joblib.load(latest_preprocessor_path)

print("\n✅ Modell und Preprocessor erfolgreich geladen.")

Lade Modell von: weights\final_catboost_regressor_20250701_004513.cbm
Lade Preprocessor von: weights\preprocessor_regression_20250701_004513.joblib

✅ Modell und Preprocessor erfolgreich geladen.


## Schritt 2: Testdaten mit der Pipeline verarbeiten

Jetzt rufen wir die Funktion `process_unlabeled_data` aus unserer Pipeline auf. Diese Funktion übernimmt das Laden, Aggregieren und Transformieren der rohen Testdaten. Sie gibt die verarbeiteten Features für das Modell und die Originaldaten (für die IDs) zurück.

In [3]:
from pathlib import Path

# Pfad zum Datenordner definieren (eine Ebene hoch, dann in den data-Ordner)
data_path = Path('../data')

# Die gesamte Datenverarbeitung wird durch einen einzigen Funktionsaufruf erledigt
X_test_processed, X_test_full = process_unlabeled_data(loaded_preprocessor, data_path)

print("\nKopf der verarbeiteten Daten:")
print(X_test_processed.head())

Starte die Verarbeitung der Testdaten mit der Pipeline...
Lade rohe Testdaten...
Aggregiere Testdaten...
Transformiere Testdaten mit dem Preprocessor...


c:\Users\lol--\miniconda3\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\lol--\miniconda3\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


✅ Datenverarbeitung abgeschlossen.

Kopf der verarbeiteten Daten:
   num_log_center__total_amount  num_log_center__n_lines  \
0                      0.919990                 0.805965   
1                     -1.474524                -0.698113   
2                      0.406978                 0.313488   
3                     -0.013416                -0.004966   
4                      0.784848                 0.805965   

   onehot_cat__cash_desk_0  onehot_cat__cash_desk_1  onehot_cat__cash_desk_2  \
0                      1.0                      0.0                      0.0   
1                      0.0                      1.0                      0.0   
2                      0.0                      0.0                      1.0   
3                      0.0                      0.0                      0.0   
4                      1.0                      0.0                      0.0   

   onehot_cat__cash_desk_3  onehot_cat__payment_medium_CASH  \
0                      0.0   

## Schritt 3: Vorhersagen erstellen

Mit den aufbereiteten Daten werden nun die Vorhersagen unter Anwendung des optimierten Schwellenwerts von **0.47** getroffen.

In [4]:
print("Erstelle Vorhersagen...")
# Wahrscheinlichkeiten für die Klasse 'FRAUD' (Klasse 1) erhalten
y_scores = loaded_model.predict_proba(X_test_processed)[:, 1]

# Den optimalen Threshold aus dem Training anwenden
optimal_threshold = 0.47
y_pred_optimized = (y_scores >= optimal_threshold).astype(int)

print(f"✅ Vorhersagen mit Threshold {optimal_threshold} abgeschlossen.")

Erstelle Vorhersagen...
✅ Vorhersagen mit Threshold 0.47 abgeschlossen.


## Schritt 4: Vorhersage-Datei erstellen und speichern

Die finalen Vorhersagen werden zusammen mit den zugehörigen Transaktions-IDs in einer CSV-Datei gespeichert.

In [6]:
# Erstelle das DataFrame für die Abgabe
# Wir verwenden X_test_full, um die ursprünglichen IDs zu erhalten
submission_df = pd.DataFrame({
    'id': X_test_full['id'],
    'prediction': y_pred_optimized
})

# Speichere die Ergebnisse als CSV-Datei
output_file = '../data/catboost_predictions_via_pipeline.csv'
submission_df.to_csv(output_file, index=False)

print(f"✅ Vorhersage-Datei '{output_file}' erfolgreich gespeichert.")
print("\nErste 5 Zeilen der Vorhersage-Datei:")
print(submission_df.head())

# Zähle die Anzahl der Vorhersagen pro Klasse
print("\nVerteilung der Vorhersagen:")
print(submission_df['prediction'].value_counts())

✅ Vorhersage-Datei '../data/catboost_predictions_via_pipeline.csv' erfolgreich gespeichert.

Erste 5 Zeilen der Vorhersage-Datei:
                                     id  prediction
0  f954103a-2669-49aa-8d64-9b7ee1f13e10           1
1  6a887829-7387-4811-98c6-0c3fd978a233           1
2  f4f334ba-878e-4573-b46e-72d7e1b3472a           1
3  d190842b-25fc-499d-b5a7-5c6758face3e           1
4  c1e02389-8ca8-4990-b261-e0983a47bdb1           1

Verteilung der Vorhersagen:
prediction
1    811617
0       288
Name: count, dtype: int64
